# 02. SASRec
Класс модели берётся из `pmixer/SASRec.pytorch`, цикл обучения и оценка свои (`common.py`, full ranking).
Вариант `ce` — полный softmax по каталогу на позициях с целью. В репозитории его нет, он написан здесь.
Готовые запуски пропускаются, ноутбук можно перезапускать после обрыва.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import gc
import importlib
import os
import sys
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

try:
    default_work_dir = '/content/drive/MyDrive/code2repo/kan_rec'
except ImportError:
    default_work_dir = os.path.abspath('./kan_rec')

WORK_DIR = os.environ.get('KAN_REC_DIR', default_work_dir)
THIRD_PARTY_DIR = os.path.join(os.path.dirname(WORK_DIR), 'third_party')
SASREC_DIR = os.path.join(THIRD_PARTY_DIR, 'SASRec.pytorch')
DATASET = 'ml1m'  # 'amazon' | 'ml1m' (для ml1m сначала 01b)

sys.path.insert(0, WORK_DIR)
import common
importlib.reload(common)

common.clone_repo('https://github.com/pmixer/SASRec.pytorch', SASREC_DIR)
sys.path.insert(0, os.path.join(SASREC_DIR, 'python'))
from model import SASRec

paths = common.dataset_paths(WORK_DIR, DATASET)
results_dir = paths['results']
os.makedirs(results_dir, exist_ok=True)
device = common.check_gpu()

SASRec.pytorch: 26a8fa8
CUDA недоступна, расчёт пойдёт на CPU


## Конфигурация
`MAX_LEN`: Amazon 50 (p99 длины истории около 71), ML-1M 200 (значение из статьи SASRec).
Бюджеты у всех конфигураций равные: одинаковые `MAX_EPOCHS` и `PATIENCE`, лимита по времени нет.
`ALLOW_NO_RANKS=True` не перезапускает готовые запуски без файла рангов, но они не попадут в бутстреп из ноутбука 05.

In [3]:
SEEDS = [0, 1, 2, 3, 4]
CONFIGS = [
    dict(loss='bce', hidden_units=64, dropout_rate=0.5),
    dict(loss='ce', hidden_units=128, dropout_rate=0.3),
    dict(loss='ce', hidden_units=128, dropout_rate=0.5),
]

MAX_LEN = dict(amazon=50, ml1m=200)[DATASET]
MAX_EPOCHS = dict(amazon=200, ml1m=400)[DATASET]
BASE_CFG = dict(num_blocks=2, num_heads=1, maxlen=MAX_LEN)

LR = 1e-3
BATCH_SIZE = 256
EVAL_EVERY = 5
PATIENCE = 4
VAL_USERS = 30000
ALLOW_NO_RANKS = True

## Данные
Левый padding, последний айтем истории стоит в конце окна.

In [4]:
data = common.load_data(paths['data'])
n_users, n_items = data['n_users'], data['n_items']
print(f'{DATASET}: users={n_users:,} items={n_items:,}')


def left_pad(items, length):
    row = np.zeros(length, np.int64)
    items = items[-length:]
    row[length - len(items):] = items
    return row


train_inputs = np.zeros((n_users, MAX_LEN), np.int64)
train_targets = np.zeros((n_users, MAX_LEN), np.int64)
val_inputs = np.zeros((n_users, MAX_LEN), np.int64)
test_inputs = np.zeros((n_users, MAX_LEN), np.int64)

for user, items in enumerate(data['train']):
    train_inputs[user] = left_pad(items[:-1], MAX_LEN)
    train_targets[user] = left_pad(items[1:], MAX_LEN)
    val_inputs[user] = left_pad(items, MAX_LEN)
    test_inputs[user] = left_pad(items + [int(data['val'][user])], MAX_LEN)


def make_score_fn(model):
    def score_fn(batch, split):
        inputs = val_inputs if split == 'val' else test_inputs
        features = model.log2feats(inputs[batch])[:, -1, :]
        return features @ model.item_emb.weight.T
    return score_fn

ml1m: users=6,040 items=3,416


## Имена запусков
Имя строится из конфигурации: `sasrec_{loss}_h{H}_d{dropout без точки}_seed{S}`. Если под таким именем лежит файл с другой
конфигурацией, он переносится в `results/_stale`. Для BCE h64 d0.5 дополнительно распознаётся старое имя `sasrec_seed{S}`.

In [5]:
def run_name(cfg, seed):
    dropout = str(cfg['dropout_rate']).replace('.', '')
    return f"sasrec_{cfg['loss']}_h{cfg['hidden_units']}_d{dropout}_seed{seed}"


def config_matches(stem, cfg):
    saved = common.load_run(results_dir, stem)
    saved_cfg = saved.get('cfg', {})
    return (
        saved.get('loss', 'bce') == cfg['loss']
        and saved_cfg.get('hidden_units') == cfg['hidden_units']
        and abs(saved_cfg.get('dropout_rate', -1) - cfg['dropout_rate']) < 1e-9
        and saved_cfg.get('maxlen') == MAX_LEN
        and saved_cfg.get('num_blocks', BASE_CFG['num_blocks']) == BASE_CFG['num_blocks']
        and saved_cfg.get('num_heads', BASE_CFG['num_heads']) == BASE_CFG['num_heads']
    )


def find_finished_run(cfg, seed):
    candidates = [run_name(cfg, seed)]
    if (cfg['loss'], cfg['hidden_units'], cfg['dropout_rate']) == ('bce', 64, 0.5):
        candidates.append(f'sasrec_seed{seed}')

    for stem in candidates:
        if not os.path.exists(f'{results_dir}/{stem}.json'):
            continue
        if config_matches(stem, cfg):
            if common.is_finished(results_dir, stem, need_ranks=not ALLOW_NO_RANKS):
                return stem
            return None
        if stem == run_name(cfg, seed):
            print(f'{stem}: в файле другая конфигурация, переношу в _stale')
            common.move_to_stale(results_dir, stem)
    return None

## Обучение

In [6]:
bce_loss = torch.nn.BCEWithLogitsLoss()


def build_model(model_cfg, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    args = SimpleNamespace(device=device, norm_first=False, **model_cfg)
    model = SASRec(n_users, n_items, args).to(device)

    for _, param in model.named_parameters():
        try:
            torch.nn.init.xavier_normal_(param.data)
        except Exception:
            pass  # одномерные параметры xavier не поддерживает
    model.pos_emb.weight.data[0, :] = 0
    model.item_emb.weight.data[0, :] = 0

    # dropout во всех модулях должен совпасть с конфигурацией (из-за этого раньше был перепутан d05)
    actual = {m.p for m in model.modules() if isinstance(m, torch.nn.Dropout)}
    actual |= {m.dropout for m in model.modules() if isinstance(m, torch.nn.MultiheadAttention)}
    assert actual == {model_cfg['dropout_rate']}, f"dropout в модели {actual}, в конфигурации {model_cfg['dropout_rate']}"
    return model


def train_epoch(model, optimizer, loss_type, rng):
    model.train()
    order = rng.permutation(n_users)
    total_loss, n_batches = 0.0, 0

    for start in range(0, n_users, BATCH_SIZE):
        batch = order[start:start + BATCH_SIZE]
        inputs, targets = train_inputs[batch], train_targets[batch]
        has_target = torch.from_numpy(targets != 0).to(device)

        if loss_type == 'bce':
            negatives = rng.randint(1, n_items + 1, size=targets.shape)
            negatives[targets == 0] = 0
            pos_logits, neg_logits = model(batch, inputs, targets, negatives)
            pos_logits, neg_logits = pos_logits[has_target], neg_logits[has_target]
            loss = (bce_loss(pos_logits, torch.ones_like(pos_logits))
                    + bce_loss(neg_logits, torch.zeros_like(neg_logits)))
        else:
            features = model.log2feats(inputs)[has_target]
            logits = features @ model.item_emb.weight.T
            logits[:, 0] = -float('inf')  # padding не может быть ответом
            target_ids = torch.from_numpy(targets).to(device)[has_target]
            loss = F.cross_entropy(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


def train_run(cfg, seed):
    model_cfg = dict(BASE_CFG, hidden_units=cfg['hidden_units'], dropout_rate=cfg['dropout_rate'])
    model = build_model(model_cfg, seed)
    n_params = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98))
    score_fn = make_score_fn(model)

    rng = np.random.RandomState(seed)
    val_users = rng.choice(n_users, size=min(VAL_USERS, n_users), replace=False)

    best_ndcg, best_epoch, bad_evals, best_state = -1, 0, 0, None
    log, epoch_times, stopped = [], [], 'max_epochs'

    for epoch in range(1, MAX_EPOCHS + 1):
        started = time.time()
        loss = train_epoch(model, optimizer, cfg['loss'], rng)
        epoch_times.append(time.time() - started)
        message = f'epoch {epoch:3d}  loss {loss:.4f}  ({epoch_times[-1]:.0f}s)'

        if epoch % EVAL_EVERY == 0:
            model.eval()
            val = common.evaluate(score_fn, data, 'val', device=device, users=val_users)
            message += f"  | val R@10 {val['Recall@10']:.4f}  N@10 {val['NDCG@10']:.4f}  R@20 {val['Recall@20']:.4f}"
            log.append(dict(epoch=epoch, loss=loss, **val))

            if val['NDCG@10'] > best_ndcg:
                best_ndcg, best_epoch, bad_evals = val['NDCG@10'], epoch, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad_evals += 1

        print(message)
        if bad_evals >= PATIENCE:
            print('early stop')
            stopped = 'early_stop'
            break

    model.load_state_dict(best_state)
    model.eval()
    val_metrics, test_metrics, val_ranks, test_ranks = common.full_evaluation(score_fn, data, device)
    print('best epoch:', best_epoch)
    print('VAL ', {k: round(v, 4) for k, v in val_metrics.items()})
    print('TEST', {k: round(v, 4) for k, v in test_metrics.items()})

    stem = run_name(cfg, seed)
    output = dict(
        model='SASRec' if cfg['loss'] == 'bce' else 'SASRec-CE',
        loss=cfg['loss'], seed=seed, cfg=model_cfg, lr=LR, batch=BATCH_SIZE,
        max_epochs=MAX_EPOCHS, patience=PATIENCE, eval_every=EVAL_EVERY,
        best_epoch=best_epoch, epochs_run=epoch, stopped=stopped,
        n_params=n_params, sec_per_epoch=float(np.mean(epoch_times)), gpu=common.gpu_name(),
        val=val_metrics, test=test_metrics, log=log,
    )
    common.save_run(results_dir, stem, output, val_ranks, test_ranks)

    del model, optimizer, best_state
    gc.collect()
    torch.cuda.empty_cache()

In [7]:
for cfg in CONFIGS:
    for seed in SEEDS:
        finished = find_finished_run(cfg, seed)
        if finished:
            print(f'[пропуск] {finished}')
            continue
        print(f'\n=== {run_name(cfg, seed)} ===')
        train_run(cfg, seed)


=== sasrec_bce_h64_d05_seed0 ===
epoch   1  loss 1.2862  (65s)


KeyboardInterrupt: 

## Сводка по val
Конфигурацию SASRec для сравнения с остальными моделями выбираем только по val.

In [ ]:
rows = []
for cfg in CONFIGS:
    for seed in SEEDS:
        candidates = [run_name(cfg, seed), f'sasrec_seed{seed}']
        for stem in candidates:
            if os.path.exists(f'{results_dir}/{stem}.json') and config_matches(stem, cfg):
                saved = common.load_run(results_dir, stem)
                rows.append(dict(
                    config=f"{cfg['loss']} h{cfg['hidden_units']} d{cfg['dropout_rate']}",
                    seed=seed, best_epoch=saved['best_epoch'], stopped=saved.get('stopped', '?'),
                    val_NDCG10=saved['val']['NDCG@10'],
                ))
                break

summary = pd.DataFrame(rows)
display(summary.groupby('config').val_NDCG10.agg(['mean', 'std', 'count']).sort_values('mean', ascending=False))